# 🤖 Modelagem Preditiva: Treinamento, Comparação e Auditoria de Modelos (Passo 6)

Este notebook apresenta a etapa de modelagem preditiva para classificar transações sob risco de detração de NPS no e-commerce. Seguindo rigorosas práticas de engenharia de machine learning (MLOps) e inferência estatística, realizaremos:
1. **Holdout Estratificado:** Divisão 80% treino e 20% teste para preservar a proporção real de detratores (74.04%).
2. **Auditoria de Algoritmos (Stratified 5-Fold Cross-Validation):** Avaliação comparativa robusta entre um Baseline trivial (`DummyClassifier`), a interpretabilidade estável da `Regressão Logística` e a flexibilidade não linear de `Random Forest`.
3. **Avaliação no Holdout:** Validação final e cega com as principais métricas de classificação (`ROC-AUC`, `Average Precision`, `Accuracy`, `F1`, `Precision` e `Recall`).
4. **Importância por Permutação (Permutation Importance):** Identificação científica dos principais fatores operacionais geradores de detração de NPS sem o viés de cardinalidade das árvores de decisão.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, precision_score, recall_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance

# Configuração visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### 1. Carregando os Dados Processados e Dividindo em Treino/Teste
Separamos de forma limpa a base processada em features explicativas ($X$) e a variável alvo ($y$), eliminando identificadores e colunas que representam vazamento de dados (*data leakage*).

In [ ]:
# Resgatar caminhos relativos ao projeto
BASE_DIR = Path(os.getcwd()).resolve()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

PROCESSED_DATA_PATH = BASE_DIR / "data" / "processed" / "processed_nps_data.csv"
if not PROCESSED_DATA_PATH.exists():
    PROCESSED_DATA_PATH = Path("processed_nps_data.csv")

df = pd.read_csv(PROCESSED_DATA_PATH)
print(f"Dados carregados. Shape: {df.shape}")

# Importando funções do nosso pipeline de features
import sys
sys.path.append(str(BASE_DIR))
from src.features import split_features_target, get_feature_lists, build_preprocessing_pipeline

X, y = split_features_target(df)

# Divisão estratificada (80% treino e 20% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

print(f"✓ Divisão efetuada!")
print(f"-> Treino: {X_train.shape[0]} amostras (Proporção Detratores: {y_train.mean()*100:.2f}%)")
print(f"-> Holdout (Teste): {X_test.shape[0]} amostras (Proporção Detratores: {y_test.mean()*100:.2f}%)")

### 2. Validação Cruzada Estratificada de 5 Folds
Avaliamos o desempenho médio e desvio padrão para certificar o poder preditivo sem overfitting. Usamos o `ColumnTransformer` configurado no passo anterior para processar as colunas de forma blindada em cada fold.

In [ ]:
num_features, cat_features = get_feature_lists(X_train)
preprocessor = build_preprocessing_pipeline(num_features, cat_features)

models = {
    "Baseline Trivial (Most Frequent)": DummyClassifier(strategy="most_frequent", random_state=42),
    "Regressão Logística (Interpretável)": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest (Não Linear)": RandomForestClassifier(random_state=42, max_depth=6)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results_list = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    scores = cross_validate(
        pipe, X_train, y_train, 
        cv=skf, 
        scoring=['roc_auc', 'average_precision', 'accuracy', 'f1'],
        return_train_score=False
    )
    
    for metric in ['roc_auc', 'average_precision', 'accuracy', 'f1']:
        cv_results_list.append({
            "Modelo": name,
            "Métrica": metric.upper(),
            "Média CV": scores[f'test_{metric}'].mean(),
            "Desvio Padrão CV": scores[f'test_{metric}'].std()
        })

df_cv = pd.DataFrame(cv_results_list)
df_cv.pivot(index="Modelo", columns="Métrica", values=["Média CV", "Desvio Padrão CV"]).round(4)

### 3. Treinamento do Modelo Campeão e Validação no Conjunto de Holdout
A Regressão Logística apresentou o melhor trade-off entre performance (ROC-AUC CV: 0.8758 e Average Precision CV: 0.9508) e interpretabilidade linear para o negócio. Vamos treiná-la na base de treino completa e validá-la cega na base de teste.

In [ ]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

final_pipeline.fit(X_train, y_train)

# Predições na base cega de teste
y_pred_proba = final_pipeline.predict_proba(X_test)[:, 1]
y_pred = final_pipeline.predict(X_test)

print("=== MÉTRICAS NO CONJUNTO DE HOLDOUT (TESTE) ===")
print(f"ROC-AUC:           {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Average Precision: {average_precision_score(y_test, y_pred_proba):.4f}")
print(f"Acurácia:          {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score:          {f1_score(y_test, y_pred):.4f}")
print(f"Precisão:          {precision_score(y_test, y_pred):.4f}")
print(f"Recall (Revocação):{recall_score(y_test, y_pred):.4f}")

### 4. Matriz de Confusão e Curvas Diagnósticas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Não Detrator", "Detrator"]).plot(
    ax=axes[0], cmap="Blues", values_format="d"
)
axes[0].set_title("Matriz de Confusão - Holdout (Teste)", fontsize=12, fontweight="bold")
axes[0].grid(False)

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {roc_auc_score(y_test, y_pred_proba):.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
axes[1].set_xlabel("Taxa de Falso Positivo (1 - Especificidade)")
axes[1].set_ylabel("Taxa de Verdadeiro Positivo (Sensibilidade)")
axes[1].set_title("Curva ROC - Holdout (Teste)", fontsize=12, fontweight="bold")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

### 5. Importância por Permutação (Rigor Científico contra Impureza Gini)
O cálculo de permutação embaralha individualmente cada feature do conjunto de holdout e avalia o impacto direto na ROC-AUC do modelo, blindando a análise de fatores distorcivos.

In [ ]:
result = permutation_importance(
    final_pipeline, X_test, y_test, 
    scoring='roc_auc', 
    n_repeats=10, 
    random_state=42
)

importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values(by='importance_mean', ascending=True)

# Plotagem horizontal
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance_mean'], xerr=importance_df['importance_std'], color="royalblue")
plt.title("Importância por Permutação de Variáveis (Holdout ROC-AUC)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Queda Média na ROC-AUC", fontsize=12)
plt.ylabel("Variável Explicativa", fontsize=12)
plt.tight_layout()
plt.show()

### 6. Salvando o Modelo Serializado
Para concluir o Passo 6, o pipeline final composto do ColumnTransformer de pré-processamento e o estimador ajustado da Regressão Logística é exportado de forma segura em arquivo binário.

In [ ]:
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "detractor_classifier.joblib"
joblib.dump(final_pipeline, MODEL_PATH)
print(f"✓ Pipeline preditivo completo salvo com sucesso em: {MODEL_PATH.resolve()}")